In [2]:
# %pip install kagglehub

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("matthewjansen/ucf101-action-recognition")

print("Path to dataset files:", path)

/home/duyth/miniconda/envs/jupyter_notebooks/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Resuming download from 3062890496 bytes (3944737769 bytes left)...
Resuming download to /home/duyth/.cache/kagglehub/datasets/matthewjansen/ucf101-action-recognition/4.archive (3062890496/7007628265) bytes left.


100%|██████████| 6.53G/6.53G [20:05<00:00, 3.27MB/s]  

Extracting files...


Path to dataset files: /home/duyth/.cache/kagglehub/datasets/matthewjansen/ucf101-action-recognition/versions/4


In [4]:
import json
import csv
from pathlib import Path
from collections import OrderedDict

# Path to the dataset
base_root = Path("/home/duyth/ai_coding/TubeViT/data/")
data_root = base_root / "ucf101"
annotations_dir = base_root / "annotations"
annotations_dir.mkdir(exist_ok=True)

# First, get all unique labels from both train and val to create consistent class mapping
print("Collecting all labels...")
all_labels = set()
for csv_file in [data_root / "train.csv", data_root / "val.csv"]:
    with open(csv_file, 'r') as f:
        reader = csv.DictReader(f)
        for row in reader:
            all_labels.add(row['label'])


In [11]:

# Create sorted label list and mapping (consistent across train/val)
sorted_labels = sorted(list(all_labels))
label_to_index = {label: idx for idx, label in enumerate(sorted_labels)}

print(f"Found {len(sorted_labels)} unique classes")
print(f"First 10 classes: {sorted_labels[:10]}")
print(f"Last 10 classes: {sorted_labels[-10:]}")


Found 101 unique classes
First 10 classes: ['ApplyEyeMakeup', 'ApplyLipstick', 'Archery', 'BabyCrawling', 'BalanceBeam', 'BandMarching', 'BaseballPitch', 'Basketball', 'BasketballDunk', 'BenchPress']
Last 10 classes: ['TennisSwing', 'ThrowDiscus', 'TrampolineJumping', 'Typing', 'UnevenBars', 'VolleyballSpiking', 'WalkingWithDog', 'WallPushups', 'WritingOnBoard', 'YoYo']


In [12]:
# Create torchvision-compatible .txt annotation files
# Torchvision expects: trainlist01.txt, testlist01.txt, classInd.txt
# Format: trainlist01.txt contains "Class/video.avi class_index" (one per line)
# Note: class_index should be 1-indexed (1-101) for torchvision

def create_txt_annotations(csv_path, output_txt_path, label_map, split_name="train"):
    """
    Create torchvision-compatible .txt annotation file from CSV.
    Format: "Class/video.avi class_index" (one per line)
    class_index is 1-indexed (1-101) for torchvision compatibility
    """
    annotations = []
    
    with open(csv_path, 'r') as f:
        reader = csv.DictReader(f)
        for row in reader:
            # Extract class and video name from clip_path
            # Format: /train/Class/video.avi or /val/Class/video.avi
            clip_path = row['clip_path'].strip()
            # Remove leading slash and split
            parts = clip_path.lstrip('/').split('/')
            if len(parts) >= 2:
                class_name = parts[1]  # e.g., "Swing"
                video_name = parts[2]  # e.g., "v_Swing_g05_c02.avi"
                # Create the line in format "Class/video.avi class_index"
                # Torchvision uses 1-indexed class indices
                class_index_1based = label_map[row['label']] + 1
                line = f"{class_name}/{video_name} {class_index_1based}"
                annotations.append(line)
    
    # Save to .txt file
    with open(output_txt_path, 'w') as f:
        f.write('\n'.join(annotations))
    
    print(f"Created {split_name} annotation file: {output_txt_path}")
    print(f"Total videos: {len(annotations)}")
    print(f"Sample lines (first 3):")
    for line in annotations[:3]:
        print(f"  {line}")
    return annotations

# Create trainlist01.txt
print("\n" + "="*50)
print("Creating trainlist01.txt...")
train_csv = data_root / "train.csv"
train_txt = annotations_dir / "trainlist01.txt"
train_lines = create_txt_annotations(train_csv, train_txt, label_to_index, "train")

# Create testlist01.txt (using val data)
print("\n" + "="*50)
print("Creating testlist01.txt...")
val_csv = data_root / "val.csv"
test_txt = annotations_dir / "testlist01.txt"
test_lines = create_txt_annotations(val_csv, test_txt, label_to_index, "test")

# Create classInd.txt (class index to name mapping)
print("\n" + "="*50)
print("Creating classInd.txt...")
classind_txt = annotations_dir / "classInd.txt"
with open(classind_txt, 'w') as f:
    # Format: "index class_name" (1-indexed for torchvision)
    for idx, label in enumerate(sorted_labels, start=1):
        f.write(f"{idx} {label}\n")
print(f"Created classInd.txt: {classind_txt}")
print(f"Total classes: {len(sorted_labels)}")

print("\n" + "="*50)
print("Summary of .txt annotation files:")
print(f"  - trainlist01.txt: {len(train_lines)} videos")
print(f"  - testlist01.txt: {len(test_lines)} videos")
print(f"  - classInd.txt: {len(sorted_labels)} classes")
print(f"\nAnnotation directory: {annotations_dir}")
print("\nUsage in train.py:")
print(f"  --dataset-root: {data_root}")
print(f"  --annotation-path: {annotations_dir}  (directory containing trainlist01.txt)")


Creating trainlist01.txt...
Created train annotation file: /home/duyth/ai_coding/TubeViT/data/annotations/trainlist01.txt
Total videos: 10055
Sample lines (first 3):
  Swing/v_Swing_g05_c02.avi 89
  Swing/v_Swing_g21_c03.avi 89
  Swing/v_Swing_g07_c01.avi 89

Creating testlist01.txt...
Created test annotation file: /home/duyth/ai_coding/TubeViT/data/annotations/testlist01.txt
Total videos: 1673
Sample lines (first 3):
  Swing/v_Swing_g22_c05.avi 89
  Swing/v_Swing_g25_c02.avi 89
  Swing/v_Swing_g06_c07.avi 89

Creating classInd.txt...
Created classInd.txt: /home/duyth/ai_coding/TubeViT/data/annotations/classInd.txt
Total classes: 101

Summary of .txt annotation files:
  - trainlist01.txt: 10055 videos
  - testlist01.txt: 1673 videos
  - classInd.txt: 101 classes

Annotation directory: /home/duyth/ai_coding/TubeViT/data/annotations

Usage in train.py:
  --dataset-root: /home/duyth/ai_coding/TubeViT/data/ucf101
  --annotation-path: /home/duyth/ai_coding/TubeViT/data/annotations  (direct